In [1]:
!pip install -q chromadb


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 99.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

In [2]:
import chromadb

client = chromadb.Client()
collection = client.get_or_create_collection(name='course_notes')

print('Collection created:', collection.name)


Collection created: course_notes


In [3]:
documents = [
    'A stack follows the Last In First Out principle.',
    'A queue follows the First In First Out principle.',
    'Binary search repeatedly divides a sorted search space.',
    'A relational database stores structured data in tables.',
    'Machine learning models learn patterns from data.'
]

ids = ['note1', 'note2', 'note3', 'note4', 'note5']

collection.add(
    ids=ids,
    documents=documents,
    metadatas=[
        {'topic': 'data_structures'},
        {'topic': 'data_structures'},
        {'topic': 'algorithms'},
        {'topic': 'databases'},
        {'topic': 'machine_learning'}
    ]
)

print('Documents inserted:', collection.count())


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:08<00:00, 9.70MiB/s]


Documents inserted: 5


In [4]:
records = collection.get()

for doc_id, doc, meta in zip(records['ids'], records['documents'], records['metadatas']):
    print(f'{doc_id}: {doc} | {meta}')


note1: A stack follows the Last In First Out principle. | {'topic': 'data_structures'}
note2: A queue follows the First In First Out principle. | {'topic': 'data_structures'}
note3: Binary search repeatedly divides a sorted search space. | {'topic': 'algorithms'}
note4: A relational database stores structured data in tables. | {'topic': 'databases'}
note5: Machine learning models learn patterns from data. | {'topic': 'machine_learning'}


In [5]:
query_text = 'How does a stack work?'

results = collection.query(
    query_texts=[query_text],
    n_results=2
)

print('Query:', query_text)
print('\nMatching documents:')
for doc in results['documents'][0]:
    print('-', doc)


Query: How does a stack work?

Matching documents:
- A stack follows the Last In First Out principle.
- A relational database stores structured data in tables.


In [6]:
filtered_results = collection.query(
    query_texts=['Which structure processes items in arrival order?'],
    n_results=2,
    where={'topic': 'data_structures'}
)

print('Filtered results:')
for doc, meta in zip(filtered_results['documents'][0], filtered_results['metadatas'][0]):
    print(doc, '->', meta)


Filtered results:
A queue follows the First In First Out principle. -> {'topic': 'data_structures'}
A stack follows the Last In First Out principle. -> {'topic': 'data_structures'}


In [7]:
collection.update(
    ids=['note5'],
    documents=['Machine learning algorithms identify useful patterns from examples and data.'],
    metadatas=[{'topic': 'machine_learning', 'level': 'intermediate'}]
)

print(collection.get(ids=['note5']))


{'ids': ['note5'], 'embeddings': None, 'documents': ['Machine learning algorithms identify useful patterns from examples and data.'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'level': 'intermediate', 'topic': 'machine_learning'}]}


In [8]:
collection.upsert(
    ids=['note3', 'note6'],
    documents=[
        'Binary search has logarithmic search complexity on a sorted collection.',
        'A graph represents relationships using vertices and edges.'
    ],
    metadatas=[
        {'topic': 'algorithms'},
        {'topic': 'data_structures'}
    ]
)

print('Record count after upsert:', collection.count())


Record count after upsert: 6


In [9]:
collection.delete(ids=['note4'])
print('Remaining records:', collection.count())


Remaining records: 5


In [10]:
collection.delete(where={'topic': 'data_structures'})
print('Records after deleting data-structure notes:', collection.count())


Records after deleting data-structure notes: 2


In [11]:
final_data = collection.get()

for doc_id, doc, meta in zip(
    final_data['ids'], final_data['documents'], final_data['metadatas']
):
    print(f'{doc_id}: {doc} | {meta}')


note3: Binary search has logarithmic search complexity on a sorted collection. | {'topic': 'algorithms'}
note5: Machine learning algorithms identify useful patterns from examples and data. | {'topic': 'machine_learning', 'level': 'intermediate'}
